## **Título:** Agente Conversacional para Admisión y Nivelación de la UG
Elaborado por:
*   Iván Andrés Moya Astudillo
*   Marco Antonio Moya Ramírez

## **Resumen:**
Diseño e implementación de un agente conversacional basado en técnicas clásicas de Procesamiento de Lenguaje Natural que, a partir de datos reales de fuentes oficiales, identifique la intención del usuario y devuelva respuestas pertinentes a preguntas frecuentes sobre admisión y nivelación de la UG, manejando adecuadamente las consultas no reconocidas.


In [ ]:
# Imports necesarios
import json # Manejo de archivos JSON
import requests # Para hacer solicitudes HTTP (descargar las intenciones y entidades definidas en github)
import re # Expresiones regulares para limpiar texto
import numpy as np # Manejo de tablas
import pandas as pd # Lectura y escritura de archivos csv
import os # Importamos el módulo 'os' para manejar rutas de archivos

#Para el tokenizado y preprocesamiento de texto
import nltk
from nltk.corpus import stopwords
!pip -q install scikit-learn pandas nltk unidecode
from unidecode import unidecode # Convierte texto con tildes a texto sin tildes
# Para obtener las raices de las palabras
from nltk.stem.snowball import SnowballStemmer
# Para vectorizar el texto
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score # Included for completeness, though mainly for evaluation

# --- NLTK Downloads and Installations ---
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
!pip -q install scikit-learn pandas nltk unidecode

# ----- Configuración de herramientas de PLN -----
stemmer = SnowballStemmer('spanish')
spanish_stopwords = set(stopwords.words('spanish'))

# ---- URLs de los archivos de datos alojados en GitHub ----
URL_INTENTS = "https://raw.githubusercontent.com/Ivn5739/Proyecto-PLN/refs/heads/main/intents.json"
URL_ENTITIES = "https://raw.githubusercontent.com/Ivn5739/Proyecto-PLN/refs/heads/main/entities.json"
URL_TEST_QUERIES = 'https://raw.githubusercontent.com/Ivn5739/Proyecto-PLN/refs/heads/main/test_queries.csv'
INTENTS_FILE_NAME = "intents.json"
ENTITIES_FILE_NAME = "entities.json"
TEST_QUERIES_FILE_NAME = 'test_queries.csv'

# ---- Carga de intents y entities en formato json ----
def load_json_from_url(url):
    # Función para descargar un archivo JSON desde una URL.
    # Realiza una petición GET a la URL y retorna el contenido parseado como JSON.
    print(f"Descargando datos desde {url}...")
    return requests.get(url).json()

def load_data(file_name, url, key=None):
    # Función para cargar datos desde un archivo local o descargarlos si no existen.
    # Prioriza la carga desde un archivo local para evitar descargas repetidas.
    data = None
    # Verifica si el archivo especificado ya existe localmente.
    if os.path.exists(file_name):
        print(f"Cargando {file_name} desde el almacenamiento local.")
        # Abre el archivo local en modo lectura y carga su contenido JSON.
        with open(file_name, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        # Si el archivo no existe localmente, lo descarga desde la URL proporcionada.
        print(f"{file_name} no encontrado localmente. Descargando desde GitHub...")
        data = load_json_from_url(url)
        # Guarda el archivo descargado localmente para futuras ejecuciones y acceso rápido.
        with open(file_name, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"{file_name} guardado localmente.")

    # Si se proporciona una 'key' y los datos son un diccionario que contiene esa 'key',
    # retorna solo el valor asociado a esa clave (útil para extraer una sección específica del JSON).
    if key and data and key in data:
        return data[key]
    # En caso contrario, retorna todos los datos cargados.
    return data

# ---- Carga de test queries en formato csv ----
def cargar_test_queries():
    # Check if the local file exists
    if os.path.exists(TEST_QUERIES_FILE_NAME):
        print(f"Cargando {TEST_QUERIES_FILE_NAME} desde el almacenamiento local.\n")
        test_queries = pd.read_csv(TEST_QUERIES_FILE_NAME)
    else:
        print(f"{TEST_QUERIES_FILE_NAME} no encontrado localmente. Descargando desde GitHub...\n")
        # Descargamos el conjunto de pruebas (CSV) directamente desde el repositorio de GitHub
        test_queries = pd.read_csv(URL_TEST_QUERIES)
        # Guardamos una copia local en el entorno de Colab para referencia
        test_queries.to_csv(TEST_QUERIES_FILE_NAME, index=False, encoding='utf-8')
        print(f"{TEST_QUERIES_FILE_NAME} guardado localmente.\n")
    return test_queries

# ---- Evaluación del Chatbot ----
def evaluate_chatbot(knowledge, vectorizer, tags, X, test_queries):
    # --- 2. EVALUACIÓN ---
    # Inicializamos listas para comparar los valores reales contra los predichos
    y_true = []
    y_pred = []
    resultados_detalle = []

    print("Evaluando consultas de prueba...\n")

    # Recorremos cada fila del DataFrame de prueba
    for index, query_row in test_queries.iterrows():
        texto = query_row["texto"]
        true_tag = query_row["intent_true"]

        # Usamos nuestra función detect_intent para obtener la predicción del modelo
        pred_tag, score = detect_intent(knowledge, texto, vectorizer, tags, X)

        # Guardamos los resultados para el cálculo de métricas y análisis posterior
        y_true.append(true_tag)
        y_pred.append(pred_tag)
        resultados_detalle.append((texto, true_tag, pred_tag, score))

    # --- 3. CÁLCULO DE MÉTRICAS ---
    # Accuracy: porcentaje de aciertos totales
    accuracy = accuracy_score(y_true, y_pred)
    # F1-macro: media aritmética de los puntajes F1 de cada clase (balance de precisión y exhaustividad)
    f1 = f1_score(y_true, y_pred, average='macro')

    print(f"=== RESULTADOS DE EVALUACIÓN ===")
    print(f"Total de consultas evaluadas: {len(test_queries)}")
    print(f"Accuracy (exactitud): {accuracy:.2%}")
    print(f"F1-macro: {f1:.2%}")
    print()

    # --- 4. ANÁLISIS DE ERRORES ---
    # Filtramos aquellas consultas donde la intención real no coincide con la predicha
    errores = [(txt, real, pred, sc) for txt, real, pred, sc in resultados_detalle if real != pred]
    if errores:
        print("=== ERRORES ENCONTRADOS ===")
        for txt, real, pred, sc in errores:
            print(f"Consulta: '{txt}'")
            print(f"    Real: {real} | Predicho: {pred} | Similitud: {sc:.2f}\n")
    else:
        print("¡No se encontraron errores!")

    # Generamos un DataFrame final para visualizar todos los casos en una tabla interactiva
    df_resultados = pd.DataFrame(resultados_detalle, columns=["Consulta", "Intención Real", "Intención Predicha", "Similitud"])
    # Guardar los resultados detallados en un archivo CSV
    df_resultados.to_csv('evaluation_results.csv', index=False, encoding='utf-8')
    print("Los resultados detallados de la evaluación han sido guardados en 'evaluation_results.csv'.")

# ----- Función completa de preprocesamiento -----
def preprocess(text):
    # 1. Conversión a minúsculas
    text = text.lower()
    # 2. Eliminación de signos de puntuación y caracteres especiales (excepto espacios)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    # 3. # Quitamos tildes: "admisión" -> "admision"
    text = unidecode(text)
    # 4. Tokenización (palabras individuales)
    tokens = nltk.word_tokenize(text, language='spanish')
    # 5. Eliminación de stopwords en español
    tokens_filtrados = [token for token in tokens if token not in spanish_stopwords]
    # 6. Stemming (reducción a raíz) – aproximación a lematización
    tokens_stem = [stemmer.stem(token) for token in tokens_filtrados]
    # Unir nuevamente para que el vectorizador TF-IDF trabaje con el texto procesado
    return ' '.join(tokens_stem)

# ----- Extracción de entidades (ejemplo simple para carreras) -----
def extract_career(knowledge, user_input=None):
    # 1. Preprocesamos la entrada del usuario para que coincida con el formato de la base de datos
    user_clean = preprocess(user_input)
    # 2. Iteramos sobre la lista de carreras disponibles en el diccionario 'knowledge'
    for carrera in knowledge['carreras']:
        # 3. Preprocesamos el nombre de la carrera actual y verificamos si está contenido en la entrada del usuario
        if preprocess(carrera['nombre']) in user_clean:
            # Si hay coincidencia, devolvemos el objeto diccionario de la carrera
            return carrera

    # 4. Si no se encuentra ninguna coincidencia tras recorrer todas las carreras, devolvemos None
    return None

# ----- Función para la detección de intenciones -----
def detect_intent(knowledge, user_input, vectorizer, tags, X, umbral=0.2):
    # Palabras clave para identificar consultas sobre duración o tiempo de estudio
    duracion_keywords = ['dura', 'duracion', 'semestre', 'semestres', 'tiempo', 'anos', 'años']

    # 1. Limpieza y preprocesamiento de la entrada del usuario
    user_clean = preprocess(user_input)

    # 2. Heurística: Si detectamos una carrera y palabras de duración, forzamos la intención
    carrera = extract_career(knowledge, user_input) # FIX: Pass knowledge here
    if carrera and any(k in user_clean for k in [preprocess(w) for w in duracion_keywords]):
        return 'consulta_duracion_carrera', 1.0

    # 3. Vectorización de la entrada usando el modelo TF-IDF entrenado
    user_vec = vectorizer.transform([user_clean])

    # 4. Cálculo de similitud de coseno contra todos los patrones conocidos
    sims = cosine_similarity(user_vec, X).flatten()

    # 5. Identificar el índice del patrón con mayor similitud
    best_idx = np.argmax(sims)

    # 6. Validar si la similitud supera el umbral de confianza definido
    if sims[best_idx] >= umbral:
        return tags[best_idx], sims[best_idx]

    # Si no supera el umbral, devolvemos 'fallback' (desconocido)
    return 'fallback', 0.0

# ----- Respuesta -----
def get_response(intents_data, knowledge, intent_tag, user_input=None):
    # 1. Buscamos la intención correspondiente en nuestra base de datos (intents_data)
    for intent in intents_data:
        if intent['tag'] == intent_tag:
            # Seleccionamos una respuesta aleatoria de la lista de respuestas posibles
            response = np.random.choice(intent['responses'])

            # 2. Lógica especial: Si la intención es consultar la duración de una carrera
            if intent_tag == 'consulta_duracion_carrera' and user_input:
                carrera = extract_career(knowledge, user_input) # FIX: Pass knowledge here
                if carrera:
                    return f"{carrera['nombre']} dura {carrera['semestres']} semestres."

            # 3. Lógica especial: Si la intención es listar carreras de un bloque específico
            if intent_tag == 'consulta_carreras_bloque' and user_input:
                user_clean = preprocess(user_input)
                # Buscamos coincidencias con los nombres de los bloques definidos en 'knowledge'
                for key, nombre in knowledge['bloques'].items():
                    key_clean = preprocess(key)  # Normalizamos la clave para la comparación
                    if key_clean in user_clean:
                        # Filtramos las carreras que pertenecen a este bloque
                        carreras_bloque = [c['nombre'] for c in knowledge['carreras'] if c['bloque'] == key]
                        return f"Carreras en {nombre}:\n- " + "\n- ".join(carreras_bloque)

                # Si no se identifica el bloque, mostramos una lista de opciones válidas
                return "La universidad de Guayaquil cuenta 59 carreras distribuidas en distintos bloques. \nLe recomendamos que consulte por el bloque o la carrera de su interés. \nLos bloques disponibles son: Salud, Administrativas, Educación, Ingeniería, Sociales, Básicas."

            # Devolvemos la respuesta genérica si no se requiere lógica de entidad adicional
            return response

    # Mensaje de error si por alguna razón no se encuentra el tag de intención
    return "Error: intención no encontrada."

# ----- Función principal (inicia todo el bloque de ejecución) -----
def main():
    # Cargamos la lista de intenciones, el diccionario de conocimiento (entidades) y las test_queries
    intents_data = load_data(INTENTS_FILE_NAME, URL_INTENTS, key='intents')
    knowledge = load_data(ENTITIES_FILE_NAME, URL_ENTITIES)
    test_queries = cargar_test_queries()

    # ----- Representación TF-IDF con unigramas y bigramas -----
    # Inicializamos listas para almacenar los patrones procesados y sus etiquetas correspondientes
    patterns = []
    tags = []

    # Recorremos cada intención en los datos cargados
    for intent in intents_data:
        # Para cada patrón de frase dentro de la intención
        for p in intent['patterns']:
            # Preprocesamos el texto (limpieza, stemming) y lo añadimos a la lista
            patterns.append(preprocess(p))
            # Guardamos el tag asociado a ese patrón
            tags.append(intent['tag'])

    # Configuramos el vectorizador TF-IDF para usar palabras sueltas (unigramas) y pares de palabras (bigramas)
    vectorizer = TfidfVectorizer(ngram_range=(1, 2))

    # Entrenamos el vectorizador y transformamos los patrones en una matriz numérica dispersa (CSR)
    X = vectorizer.fit_transform(patterns)

    # Mensaje de bienvenida al iniciar el programa
    print("Chatbot de admisión UG. Escribe 'salir' para terminar.")

    # Ciclo infinito para mantener la conversación activa
    while True:
        # Captura la entrada de texto del usuario
        entrada = input("\nTú: ")

        # Condición de salida: si el usuario escribe palabras clave, termina el programa
        if entrada.lower() in ['salir', 'exit', 'quit']:
            print("Chatbot: ¡Hasta pronto!")
            break

        # 1. Detectar la intención y obtener el puntaje de confianza (similitud)
        tag, score = detect_intent(knowledge, entrada, vectorizer, tags, X)

        # Imprimir información de depuración para ver qué intención detectó el sistema
        print(f"[Debug: intención={tag}, similitud={score:.2f}]")

        # 2. Obtener la respuesta adecuada basándose en la intención y el texto original
        respuesta = get_response(intents_data, knowledge, tag, entrada)

        # Mostrar la respuesta final al usuario
        print(f"Chatbot:  {respuesta}")

    # Evaluacion del chatbot
    print("\nEvaluación final de la precisión del chatbot.")
    evaluate_chatbot(knowledge, vectorizer, tags, X, test_queries)

# Punto de entrada estándar de Python para ejecutar la función main
if __name__ == "__main__":
    main()

## **Fuentes:**
**Proceso de Admisión:** https://admision.ug.edu.ec/admision/

**Curso de Nivelación:** https://admision.ug.edu.ec/nivelacion/

**Calendario Académico:** https://admision.ug.edu.ec/calendario/

**Carreras universitarias:** https://admision.ug.edu.ec/

**Respositorio en Github con archivos auxiliares:** https://github.com/Ivn5739/Proyecto_PLN_P2